# Customer EDA for Segmentation

This notebook explores the cleaned customer data for the customer segmentation project. The goal is to understand customer characteristics and behavior before the group chooses features for PCA and K-Means clustering.

The cleaning work was already completed in `data_cleaning.ipynb`. The original dataset had 2,240 customers, and the cleaned dataset has 2,212 customers after preprocessing. This notebook uses `marketing_campaign_cleaned.csv` and does not repeat the cleaning process.

The data is also not scaled here because the variables are easier to interpret in their original units during EDA.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Load the Cleaned Data

The cleaned dataset is loaded from the project root. This file already includes the preprocessing decisions from the data cleaning notebook.

In [ ]:
df = pd.read_csv("../marketing_campaign_cleaned.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

The dataset contains customer demographics, spending by product type, purchase channel activity, campaign response variables, and a few useful summary columns created during cleaning.

## 2. Customer Distributions

First, I look at the main variables that could help separate different types of customers.

In [ ]:
main_vars = ["Income", "Age", "Recency", "Children", "TotalSpending", "TotalPurchases"]
df[main_vars].describe().round(2)

In [ ]:
df[main_vars].hist(figsize=(12, 8), bins=30)
plt.suptitle("Distributions of Main Customer Variables", y=1.02)
plt.tight_layout()
plt.show()

Income and total spending are not evenly distributed. Total spending is especially skewed, meaning many customers spend relatively little while a smaller group spends much more. Age is more concentrated, and most customers have between 0 and 2 children. Recency is spread fairly widely, so customers differ in how recently they bought something.

In [ ]:
product_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

product_data = df[product_cols].melt(var_name="Product", value_name="Spending")

plt.figure(figsize=(10, 5))
sns.boxplot(data=product_data, x="Spending", y="Product")
plt.title("Distribution of Spending by Product Type")
plt.xlabel("Amount Spent")
plt.ylabel("Product Type")
plt.tight_layout()
plt.show()

Product spending varies a lot by category. Wine and meat spending have much wider ranges than the other product groups, so they may be useful for describing higher-value customers or different product preferences.

In [ ]:
channel_cols = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases",
    "NumDealsPurchases",
    "NumWebVisitsMonth"
]

channel_data = df[channel_cols].melt(var_name="Channel", value_name="Count")

plt.figure(figsize=(10, 5))
sns.boxplot(data=channel_data, x="Count", y="Channel")
plt.title("Distribution of Purchase and Visit Variables")
plt.xlabel("Count")
plt.ylabel("Channel Variable")
plt.tight_layout()
plt.show()

Purchasing activity also differs across customers. Store purchases are usually the most common purchase channel, while catalog purchases tend to be lower. Web visits are frequent, but visits do not necessarily mean purchases.

## 3. Relationships Between Customer Characteristics

Next, I explore a few relationships that could matter for segmentation.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Income", y="TotalSpending", alpha=0.5)
plt.title("Income vs Total Spending")
plt.xlabel("Income")
plt.ylabel("Total Spending")
plt.tight_layout()
plt.show()

Customers with higher income generally appear to spend more. This suggests that income and customer value are related, although they are not exactly the same thing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=df, x="Age", y="TotalSpending", alpha=0.5, ax=axes[0])
axes[0].set_title("Age vs Total Spending")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Total Spending")

sns.scatterplot(data=df, x="Recency", y="TotalSpending", alpha=0.5, ax=axes[1])
axes[1].set_title("Recency vs Total Spending")
axes[1].set_xlabel("Recency")
axes[1].set_ylabel("Total Spending")

plt.tight_layout()
plt.show()

Age does not show as strong a pattern with total spending as income does. Recency also does not show a very clear relationship with total spending, but it may still be useful because it describes how recently the customer interacted with the company.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Children", y="TotalSpending")
plt.title("Children vs Total Spending")
plt.xlabel("Number of Children at Home")
plt.ylabel("Total Spending")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Children")["TotalSpending"].mean().round(2)

Customers with fewer children seem to spend more on average. This may help separate customers by household situation and available spending capacity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=df, x="NumWebPurchases", y="NumStorePurchases", alpha=0.5, ax=axes[0])
axes[0].set_title("Web Purchases vs Store Purchases")
axes[0].set_xlabel("Web Purchases")
axes[0].set_ylabel("Store Purchases")

sns.scatterplot(data=df, x="NumCatalogPurchases", y="NumStorePurchases", alpha=0.5, ax=axes[1])
axes[1].set_title("Catalog Purchases vs Store Purchases")
axes[1].set_xlabel("Catalog Purchases")
axes[1].set_ylabel("Store Purchases")

plt.tight_layout()
plt.show()

The purchasing channels have some relationship with each other, but they are not identical. Some customers buy more through stores, while others may be more active through web or catalog purchases. This could be useful for interpreting channel preference.

## 4. Product and Channel Behavior

This section compares what customers buy and how they buy it.

In [ ]:
product_means = df[product_cols].mean().sort_values()

plt.figure(figsize=(8, 5))
sns.barplot(x=product_means.values, y=product_means.index)
plt.title("Average Spending by Product Type")
plt.xlabel("Average Amount Spent")
plt.ylabel("Product Type")
plt.tight_layout()
plt.show()

Wine has the highest average spending, followed by meat products. Fruit, fish, sweets, and gold products are much lower on average. This means product categories may be helpful for explaining what different customer groups prefer.

In [ ]:
channel_means = df[channel_cols].mean().sort_values()

plt.figure(figsize=(8, 5))
sns.barplot(x=channel_means.values, y=channel_means.index)
plt.title("Average Purchase and Visit Activity")
plt.xlabel("Average Count")
plt.ylabel("Channel Variable")
plt.tight_layout()
plt.show()

Store purchases are the largest purchase channel on average, while web purchases are also important. Catalog purchases are lower, but they may still describe a specific type of customer. Deal purchases can help identify customers who are more promotion-sensitive.

## 5. Correlation and Potential Redundancy

A correlation heatmap can show which variables contain similar information. This matters because K-Means is distance-based. If many variables describe the same concept, that concept can accidentally get too much weight in the clustering.

In [ ]:
corr_cols = [
    "Income",
    "Age",
    "Recency",
    "Children",
    "TotalSpending",
    "TotalPurchases",
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases",
    "NumDealsPurchases",
    "NumWebVisitsMonth"
]

corr = df[corr_cols].corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Possible Clustering Variables")
plt.tight_layout()
plt.show()

`TotalSpending` is strongly related to several product spending columns, especially wine and meat spending. This makes sense because total spending is built from the product columns. `TotalPurchases` is also strongly related to web, catalog, and store purchases because it is calculated from them.

This does not mean these variables are wrong to use. It means the group should be careful about using both total variables and all of their detailed parts at the same time, because spending or purchase activity could be counted more than once in the clustering.

## 6. Recommendations for Final Clustering Features

These recommendations are based on the EDA above. They are meant to support the group discussion before PCA and K-Means, not to make the final decision alone.

### Potential clustering features

`Income` looks useful because it has a clear relationship with customer spending and represents financial situation.

`TotalSpending` looks useful because it summarizes customer value and has a wide range across customers.

`TotalPurchases` looks useful because it summarizes purchasing activity across the main buying channels.

`Recency` may be useful because it describes how recently customers interacted with the company, even though it does not strongly relate to total spending.

`Age` may be useful as a simple demographic variable, although it seems less strongly connected to spending than income.

`Children` may be useful because customers with fewer children appear to spend more on average, so household composition could help define customer similarity.

Some channel variables, such as `NumWebPurchases`, `NumCatalogPurchases`, and `NumStorePurchases`, may be useful if the group wants clusters to reflect how customers prefer to buy.

### Potentially redundant features

`TotalSpending` overlaps with `MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, and `MntGoldProds`. If all of these are included together, spending behavior may have too much influence on K-Means.

`TotalPurchases` overlaps with `NumWebPurchases`, `NumCatalogPurchases`, and `NumStorePurchases`. The group should decide whether it wants one overall purchase activity variable or more detailed channel behavior.

A simple option would be to start with broader variables like `Income`, `Age`, `Recency`, `Children`, `TotalSpending`, and `TotalPurchases`. A more detailed option would be to replace `TotalSpending` or `TotalPurchases` with selected product or channel variables if the group wants more behavioral detail.

### Variables better suited for cluster interpretation

The campaign response variables, including `AcceptedCmp1`, `AcceptedCmp2`, `AcceptedCmp3`, `AcceptedCmp4`, `AcceptedCmp5`, and `Response`, may be better used after clustering. They can help check whether the final customer groups respond differently to marketing campaigns.

The detailed product spending variables may also be very useful after clustering. For example, after clusters are created, the group can compare wine, meat, fish, sweets, fruit, and gold spending to describe what each cluster tends to buy.

Overall, the strongest candidates for defining similarity are variables connected to income, customer value, purchase activity, household composition, recency, and possibly channel preference.